# Blender Python API drift across language models

Blender's Python API (`bpy`) changes with every release: `scene.objects.link` disappeared in 2.80, context override
dicts in 4.0, `use_auto_smooth` in 4.1, the EEVEE engine id was renamed in 4.2 and renamed back in 5.0, the fast
boolean solver became `FLOAT` in 5.0. Models trained on a mix of tutorials from every era tend to answer with whatever
version was most common in their data.

This benchmark asks the same scripting question for Blender **3.6, 4.2, 4.5 and 5.0** and grades each answer two ways:

| axis | verdict |
|------|---------|
| **runs** | the script, followed by a case-specific assert, exits 0 inside that exact Blender build (`blender -b --factory-startup --python`) |
| **aware** | the answer's `WATCH OUT` section names every API that was removed, renamed or changed for that version, and its replacement |

The primary leaderboard task is **runs**. Awareness is reported alongside it because the two come apart:
code can work while the model has no idea the API moved, and a model can describe the change and still emit the old call.

Grading code, case bank and the list of verified API changes: [bpy-drift-bench](https://github.com/Rustam335/bpy-drift-bench).

## 1. Setup

Installs the grading library and the shared libraries a headless Linux Blender still links against.

In [ ]:
import os, sys, subprocess, platform, pathlib

ON_KAGGLE = pathlib.Path("/kaggle").exists()
REPO_URL = "https://github.com/Rustam335/bpy-drift-bench"

if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO_URL}"], check=True)
    subprocess.run("apt-get install -y -qq libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libegl1 libsm6 libxkbcommon0 > /dev/null",
                   shell=True, check=False)

import pandas as pd
import kaggle_benchmarks as kbench
from bpy_drift import (RELEASES, ensure_blender, verify_build, run_script, load_cases, expand,
                       SYSTEM_PROMPT, build_user_prompt, grade)
from bpy_drift import report

pd.set_option("display.max_colwidth", 120)
print("platform:", platform.platform())

## 2. Gate: every target Blender build starts headless

Nothing below runs until all four builds are downloaded, report the expected version, and can both pass and fail an assert. Roughly 350 MB per version.

In [ ]:
PROBE = "import bpy\nbpy.data.objects['Cube'].location.x = 1.0"
PASSING = "import bpy\nassert bpy.data.objects['Cube'].location.x == 1.0"
FAILING = "import bpy\nassert False, 'deliberate failure'"

BLENDER = {}
rows = []
for version in RELEASES:
    binary = ensure_blender(version)
    build = verify_build(binary, version)
    ok, bad = run_script(binary, PROBE, PASSING), run_script(binary, PROBE, FAILING)
    BLENDER[version] = binary
    rows.append({"target": version, "build": build, "assert passes": ok.passed,
                 "assert failure detected": not bad.passed, "failure line": bad.reason})
gate = pd.DataFrame(rows).set_index("target")
display(gate)
assert gate["assert passes"].all() and gate["assert failure detected"].all(), "a Blender build is not usable; stop here"

## 3. Case bank

Each case is one question, asked for every version it applies to. Which API changes are live traps for a case depends on the version (`changedIn <= target`). Where the correct end state itself differs between versions (EEVEE id, solver name, Principled socket names) the assert script is overridden per version.

In [ ]:
CASES = {c.id: c for c in load_cases()}
eval_df = expand(CASES.values())
print(f"{len(CASES)} cases x versions = {len(eval_df)} prompts per model")
display(eval_df.pivot_table(index="category", columns="version", values="case_id", aggfunc="count", fill_value=0, observed=False))
display(eval_df[["case_id", "category", "question"]].drop_duplicates("case_id").set_index("case_id"))

## 4. The prompt

Every model gets the same system prompt and the same user message; only the version number changes. No tools, no retrieval, temperature 0.

In [ ]:
print(SYSTEM_PROMPT)
print("-" * 60)
print(build_user_prompt("4.2", CASES["eevee-engine"].question))

## 5. Task definition

The task returns `runs`. Every graded answer, with its script, failure line and awareness verdict, is kept in `RECORDS` for the analysis below.

In [ ]:
RECORDS = []

def ask(llm, version, question):
    with kbench.chats.new(name=f"bpy {version}", system_instructions=SYSTEM_PROMPT):
        return llm.prompt(build_user_prompt(version, question), temperature=0)

@kbench.task(
    name="bpy_drift_runs",
    description="Does the model's Blender Python script run in the exact Blender version it was asked for?",
)
def bpy_drift_runs(llm, case_id: str, version: str) -> bool:
    case = CASES[case_id]
    answer = ask(llm, version, case.question)
    result = grade(answer, case, version, binary=BLENDER[version])
    RECORDS.append({"model": llm.name, "category": case.category, "answer": answer, **result.as_dict()})
    return result.runs

## 6. Dry run: one case, one model

In [ ]:
run = bpy_drift_runs.run(llm=kbench.llm, case_id="eevee-engine", version="5.0")
last = RECORDS[-1]
print("runs:", last["runs"], "| aware:", last["aware"], "| build:", last["blender_build"])
print("failures:", last["failures"])
print(last["script"])

## 7. Models

The models available to this notebook. The list is fixed before the full run and reported in the write-up; aim for a spread of vendors and sizes.

In [ ]:
print("\n".join(sorted(kbench.llms)))
MODELS = [kbench.llm]  # replace with an explicit list, e.g. [kbench.llms["google/gemini-2.5-flash"], ...]

## 8. Full evaluation

`n_jobs=1`: every grade launches a Blender process. The response cache means a re-run after a grading fix does not re-prompt the models.

In [ ]:
RECORDS.clear()
with kbench.client.enable_cache():
    runs = bpy_drift_runs.evaluate(
        llm=MODELS,
        evaluation_data=eval_df[["case_id", "version"]],
        on_failure="continue",
        max_attempts=2,
        n_jobs=1,
    )
print(f"completed: {len(runs.completed_runs)}  errored: {len(runs.errored_runs)}")

## 9. Results

In [ ]:
df = report.records_frame(RECORDS)
df.drop(columns=["answer", "stderr"]).to_csv("results.csv", index=False)
df.to_json("records.jsonl", orient="records", lines=True)

pct = "{:.0%}"
print("Run rate by model and version")
display(report.rate_table(df, "runs").style.format(pct))
print("Awareness rate by model and version")
display(report.rate_table(df, "aware").style.format(pct))
print("Where the two axes disagree")
display(report.gap_table(df).style.format({c: pct for c in ["runs", "aware", "runs but unaware", "aware but breaks"]}))
print("Most common failure lines")
display(report.failure_reasons(df))

In [ ]:
import matplotlib.pyplot as plt

for metric in ("runs", "aware"):
    report.plot_drift_curves(df, metric)
    plt.tight_layout(); plt.savefig(f"drift_{metric}.png", bbox_inches="tight"); plt.show()
report.plot_category_heatmap(df, "runs")
plt.tight_layout(); plt.savefig("heatmap_runs.png", bbox_inches="tight"); plt.show()

## 10. Publish

Keeps only the primary task's files in the working directory, so the Kaggle benchmark is built from `bpy_drift_runs`.

In [ ]:
%choose bpy_drift_runs